## 1. Setup & Imports

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=UserWarning)
np.random.seed(42)

# Add src/ to path so we can import project modules
sys.path.insert(0, str(Path.cwd().parent))

from src.data_generator import MaterialDataGenerator
from src.models import BayesianStrengthModel, HierarchicalStrengthModel
from src.visualization import (
    plot_calibration,
    plot_failure_probability,
    plot_group_comparison,
    plot_posterior,
    plot_predictive_intervals,
    plot_tractability_comparison,
)

# Results directory for figure output
RESULTS_DIR = Path.cwd().parent / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print(f"PyMC   : {pm.__version__}")
print(f"ArviZ  : {az.__version__}")
print(f"NumPy  : {np.__version__}")
print(f"Results: {RESULTS_DIR.resolve()}")

## Data Generation

In [ ]:
gen = MaterialDataGenerator(random_seed=42)

# Single-phase dataset (ferritic) for Model 1
ds_single = gen.generate_single_phase(n_samples=200, alloy="ferritic")
df = ds_single.data
true_params = ds_single.true_params

print(ds_single)
print("\nGround-truth parameters:")
for k, v in true_params.items():
    print(f"  {k:10s}: {v}")

print("\nDataset summary:")
df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Strength vs Temperature
axes[0].scatter(df["temperature"], df["tensile_strength"],
                s=18, alpha=0.55, color="#2a6496")
axes[0].set_xlabel("Temperature (°C)")
axes[0].set_ylabel("Tensile Strength (MPa)")
axes[0].set_title("Strength vs. Temperature")

# Strength vs Carbon Content
axes[1].scatter(df["carbon_content"], df["tensile_strength"],
                s=18, alpha=0.55, color="#d35400")
axes[1].set_xlabel("Carbon Content (wt%)")
axes[1].set_ylabel("Tensile Strength (MPa)")
axes[1].set_title("Strength vs. Carbon Content")

# Strength distribution
axes[2].hist(df["tensile_strength"], bins=30, color="#27ae60",
             edgecolor="white", alpha=0.8)
axes[2].axvline(df["tensile_strength"].mean(), color="black",
                ls="--", lw=1.5, label=f"Mean = {df['tensile_strength'].mean():.1f}")
axes[2].set_xlabel("Tensile Strength (MPa)")
axes[2].set_ylabel("Count")
axes[2].set_title("Strength Distribution")
axes[2].legend()

sns.despine()
fig.suptitle("Exploratory Data Analysis — Ferritic Steel", y=1.02, fontsize=13)
fig.savefig(RESULTS_DIR / "01_data_exploration.png", dpi=300, bbox_inches="tight")
plt.show()
print("Correlation with temperature:",
      df["tensile_strength"].corr(df["temperature"]).round(3))
print("Correlation with carbon content:",
      df["tensile_strength"].corr(df["carbon_content"]).round(3))

## (Model 1) Bayesian Linear Regression

In [ ]:
# Build and inspect model
model1 = BayesianStrengthModel(df)
model1.build()

print("Model 1 — Bayesian Linear Regression")
print("Variables:", [v.name for v in model1.model.free_RVs])
print("Observed:", [v.name for v in model1.model.observed_RVs])

# Model graph (requires graphviz)
try:
    graph = pm.model_to_graphviz(model1.model)
    graph
except Exception:
    print("(graphviz not available — install graphviz to render model graph)")

In [ ]:
with model1.model:
    prior_pc = pm.sample_prior_predictive(samples=500, random_seed=42)

prior_y = prior_pc.prior_predictive["bayesian_lr::y_obs"].values.flatten()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(prior_y, bins=60, density=True, color="#95a5a6",
        edgecolor="white", alpha=0.7, label="Prior predictive")
ax.hist(df["tensile_strength"], bins=30, density=True, color="#2a6496",
        edgecolor="white", alpha=0.6, label="Observed data")
ax.axvline(400, color="red", ls="--", lw=1, label="400 MPa threshold")
ax.set_xlabel("Tensile Strength (MPa)")
ax.set_ylabel("Density")
ax.set_title("Prior Predictive Check — Model 1")
ax.set_xlim(-200, 1200)
ax.legend()
sns.despine()
fig.savefig(RESULTS_DIR / "01b_prior_predictive.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Prior predictive 5th–95th percentile: "
      f"[{np.percentile(prior_y, 5):.0f}, {np.percentile(prior_y, 95):.0f}] MPa")

In [ ]:
t0_nuts = time.time()
idata1 = model1.sample(
    draws=2000,
    tune=1000,
    chains=4,
    target_accept=0.9,
    random_seed=42,
)
t_nuts = time.time() - t0_nuts
print(f"\nNUTS sampling completed in {t_nuts:.1f}s")

In [ ]:
# Posterior summary
summary_df = model1.summary()

print("\n--- Convergence Assessment ---")
for name, row in summary_df.iterrows():
    if "r_hat" in summary_df.columns:
        rhat = row.get("r_hat", float("nan"))
        ess = row.get("ess_bulk", float("nan"))
        status = "✓" if rhat < 1.01 else "⚠ CHECK"
        print(f"  {name:30s}  R̂={rhat:.4f} {status}  ESS_bulk={ess:.0f}")

In [ ]:
# Trace plots
var_names_model1 = ["bayesian_lr::alpha", "bayesian_lr::beta_T",
                    "bayesian_lr::beta_C", "bayesian_lr::sigma"]

axes = az.plot_trace(
    idata1, var_names=var_names_model1,
    figsize=(12, 8), combined=False
)
plt.suptitle("NUTS Trace Plots — Model 1 (Bayesian Linear Regression)",
             y=1.01, fontsize=13)
plt.savefig(RESULTS_DIR / "02_model1_trace.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Pair plot — joint posterior structure
axes = az.plot_pair(
    idata1,
    var_names=var_names_model1,
    divergences=True,
    figsize=(9, 9),
    kind="kde",
    textsize=9,
)
plt.suptitle("Joint Posterior — Model 1 (pair plot)", y=1.01)
plt.savefig(RESULTS_DIR / "03_model1_pairs.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Posterior distributions with ground-truth overlays
fig, axes = plot_posterior(
    idata1,
    var_names=var_names_model1,
    true_values={
        "bayesian_lr::alpha": true_params["alpha"],
        "bayesian_lr::beta_T": true_params["beta_T"],
        "bayesian_lr::beta_C": true_params["beta_C"],
        "bayesian_lr::sigma": true_params["sigma"],
    },
    title="Posterior Distributions — Model 1",
    save_path=RESULTS_DIR / "03b_model1_posteriors.png",
)
plt.show()

In [ ]:
posterior = idata1.posterior

params_to_report = [
    ("bayesian_lr::alpha", "α", "MPa", true_params["alpha"]),
    ("bayesian_lr::beta_T", "β_T", "MPa/°C", true_params["beta_T"]),
    ("bayesian_lr::beta_C", "β_C", "MPa/wt%C", true_params["beta_C"]),
    ("bayesian_lr::sigma", "σ", "MPa", true_params["sigma"]),
]

print(f"{'Parameter':>6}  {'True':>8}  {'Post. Mean':>12}  {'Post. SD':>10}  {'94% HDI':>20}  {'Recovery'}")  
print("-" * 80)
for key, sym, unit, true_val in params_to_report:
    samples = posterior[key].values.flatten()
    mean = samples.mean()
    sd   = samples.std()
    hdi  = az.hdi(samples, hdi_prob=0.94)
    in_hdi = "✓" if hdi[0] <= true_val <= hdi[1] else "✗"
    print(f"{sym:>6}  {true_val:>8.3f}  {mean:>12.4f}  {sd:>10.4f}  "
          f"[{hdi[0]:.3f}, {hdi[1]:.3f}]  {in_hdi}")

## Tractability Study NUTS vs. ADVI

In [ ]:
# Run ADVI on the same model
t0_advi = time.time()
idata1_advi = model1.sample_advi(n_iterations=50_000, random_seed=42)
t_advi = time.time() - t0_advi

print(f"ADVI completed in {t_advi:.1f}s  (NUTS took {t_nuts:.1f}s)")
print(f"Speed-up factor: {t_nuts / max(t_advi, 0.01):.1f}×")

In [ ]:
# Compare posteriors
scalar_vars = ["alpha", "beta_T", "beta_C", "sigma"]

fig, axes = plot_tractability_comparison(
    idata_nuts=idata1,
    idata_advi=idata1_advi,
    var_names=scalar_vars,
    time_nuts=t_nuts,
    time_advi=t_advi,
    save_path=RESULTS_DIR / "04_tractability_comparison.png",
)
plt.show()

In [ ]:
# Quantitative comparison table
print(f"{'Param':>10} {'NUTS Mean':>12} {'ADVI Mean':>12} {'NUTS SD':>10} "
      f"{'ADVI SD':>10} {'SD ratio (ADVI/NUTS)':>20}")
print("-" * 80)

for v in scalar_vars:
    for nuts_key in [f"bayesian_lr::{v}", v]:
        if nuts_key in idata1.posterior:
            nuts_s = idata1.posterior[nuts_key].values.flatten()
            break
    for advi_key in [f"bayesian_lr::{v}", v]:
        if advi_key in idata1_advi.posterior:
            advi_s = idata1_advi.posterior[advi_key].values.flatten()
            break

    ratio = advi_s.std() / max(nuts_s.std(), 1e-9)
    print(f"{v:>10} {nuts_s.mean():>12.4f} {advi_s.mean():>12.4f} "
          f"{nuts_s.std():>10.4f} {advi_s.std():>10.4f} {ratio:>20.3f}")

print("\nNote: ADVI SD/NUTS SD < 1 indicates mean-field underestimation of posterior variance.")

## (Model 2) Hierarchical Bayesian Model


In [ ]:
# Generate multi-phase dataset
ds_multi = gen.generate_multiphase(
    n_per_group={"austenitic": 80, "ferritic": 100, "martensitic": 60}
)
df_multi = ds_multi.data
true_params_multi = ds_multi.true_params

print(ds_multi)
print(f"\nGroup counts:\n{df_multi['alloy_phase'].value_counts().to_string()}")

# Visualise multi-group data
fig, ax = plt.subplots(figsize=(8, 5))
palette = {"austenitic": "#2a6496", "ferritic": "#d35400", "martensitic": "#27ae60"}
for phase, grp in df_multi.groupby("alloy_phase"):
    ax.scatter(grp["temperature"], grp["tensile_strength"],
               s=18, alpha=0.5, color=palette[phase], label=phase.capitalize())
ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Tensile Strength (MPa)")
ax.set_title("Multi-Phase Steel Dataset — Strength vs. Temperature")
ax.legend(title="Alloy Phase")
sns.despine()
fig.savefig(RESULTS_DIR / "04b_multiphase_data.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Build and sample hierarchical model
hm = HierarchicalStrengthModel(
    df_multi,
    group_labels=["austenitic", "ferritic", "martensitic"],
)
hm.build()

print("Free RVs:", [v.name for v in hm.model.free_RVs])

idata_h = hm.sample(
    draws=2000,
    tune=1000,
    chains=4,
    target_accept=0.9,
    random_seed=42,
)
print("\nSampling complete.")

In [ ]:
# Hierarchical model convergence diagnostics
hm.summary()

In [ ]:
# Fit separate (no-pooling) models for comparison
idata_list_np = []
for phase in ["austenitic", "ferritic", "martensitic"]:
    df_phase = df_multi[df_multi["alloy_phase"] == phase].copy()
    m = BayesianStrengthModel(df_phase, name=f"bayesian_lr_{phase}")
    m.build()
    idata_np = m.sample(draws=2000, tune=1000, chains=2, target_accept=0.9,
                        random_seed=42)
    idata_list_np.append(idata_np)
    print(f"{phase}: sampling done")

In [ ]:
true_alphas = {
    "austenitic": true_params_multi["austenitic"]["alpha"],
    "ferritic":   true_params_multi["ferritic"]["alpha"],
    "martensitic": true_params_multi["martensitic"]["alpha"],
}

fig, ax = plot_group_comparison(
    idata_hierarchical=idata_h,
    idata_no_pool=idata_list_np,
    group_labels=["Austenitic", "Ferritic", "Martensitic"],
    true_alphas=true_alphas,
    save_path=RESULTS_DIR / "05_hierarchical_shrinkage.png",
)
plt.show()

## Posterior Predictive Analysis

In [ ]:
# Posterior predictive for Model 1 over temperature grid
T_grid = np.linspace(20, 300, 100)
C_ref = np.full_like(T_grid, 0.30)  # fixed at median carbon content

pred_samples = model1.predict(
    new_temperature=T_grid,
    new_carbon=C_ref,
    n_samples=4000,
)

fig, ax = plot_predictive_intervals(
    temperature_grid=T_grid,
    predictive_samples=pred_samples,
    observed_temperature=df["temperature"].values,
    observed_strength=df["tensile_strength"].values,
    carbon_ref=0.30,
    title="Posterior Predictive — Model 1 (Bayesian LR)",
    save_path=RESULTS_DIR / "06_predictive_intervals.png",
)
plt.show()

In [ ]:
# ArviZ PPC plot
axes = az.plot_ppc(
    idata1,
    num_pp_samples=200,
    figsize=(8, 4),
    alpha=0.3,
)
plt.title("Posterior Predictive Check — Model 1")
plt.savefig(RESULTS_DIR / "06b_ppc.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Hold out 20% of data and evaluate calibration
rng_split = np.random.default_rng(0)
n_test = int(0.2 * len(df))
test_idx = rng_split.choice(len(df), size=n_test, replace=False)
train_idx = np.setdiff1d(np.arange(len(df)), test_idx)

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test  = df.iloc[test_idx].reset_index(drop=True)

# Fit on training split
model1_cv = BayesianStrengthModel(df_train, name="bayesian_lr_cv")
model1_cv.build()
idata_cv = model1_cv.sample(draws=2000, tune=500, chains=2, target_accept=0.9,
                             random_seed=42)

# Predict on test split
pred_test = model1_cv.predict(
    new_temperature=df_test["temperature"].values,
    new_carbon=df_test["carbon_content"].values,
    n_samples=4000,
)

fig, ax = plot_calibration(
    observed=df_test["tensile_strength"].values,
    predictive_samples=pred_test,
    n_levels=20,
    save_path=RESULTS_DIR / "07_calibration.png",
)
plt.show()

In [ ]:
fig, ax = plot_failure_probability(
    temperature_grid=T_grid,
    predictive_samples=pred_samples,
    threshold_mpa=400.0,
    carbon_ref=0.30,
    save_path=RESULTS_DIR / "08_failure_probability.png",
)
plt.show()

# Report at specific temperatures
print("Failure probability P(σ_y < 400 MPa | T, C=0.30 wt%):")
for T_query in [100, 200, 300]:
    idx = np.argmin(np.abs(T_grid - T_query))
    p_fail = (pred_samples[:, idx] < 400).mean()
    print(f"  T = {T_query:3d} °C  →  P_fail = {p_fail:.4f} ({100*p_fail:.1f}%)")

In [ ]:
try:
    loo1 = az.loo(idata1, pointwise=True)
    print("Model 1 (Bayesian LR) — LOO-CV:")
    print(f"  ELPD_LOO = {loo1.elpd_loo:.2f} ± {loo1.se:.2f}")
    print(f"  p_LOO    = {loo1.p_loo:.2f}  (effective number of parameters)")
except Exception as e:
    print(f"LOO computation skipped: {e}")

print()
print("--- Summary of All Models ---")
print(f"{'Model':30s} {'N':>6} {'Features':>25} {'Inference':>12}")
print("-" * 80)
models_summary = [
    ("Bayesian LR (NUTS)",     200, "T + C, single phase",     "NUTS"),
    ("Bayesian LR (ADVI)",     200, "T + C, single phase",     "ADVI"),
    ("Hierarchical (NUTS)",    240, "T + C + phase (partial pool)", "NUTS"),
]
for name, n, feats, inf_method in models_summary:
    print(f"{name:30s} {n:>6} {feats:>25} {inf_method:>12}")

In [ ]:
# Print final summary of all output files
print("=" * 50)
print("OUTPUT FILES WRITTEN TO results/")
print("=" * 50)
for f in sorted(RESULTS_DIR.glob("*.png")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s}  ({size_kb:.0f} KB)")